# Imports and functions

In [ ]:
import jax
import jax.numpy as jnp

from jax import random, jit, lax, grad, value_and_grad, jacobian, jacfwd, jacrev, hessian, vmap, ops, tree_util, debug, config
config.update('jax_enable_x64', True)
from jax.experimental import checkify
from jax_md import space, smap, energy, minimize, quantity, simulate, util
from jaxopt import LBFGS, ScipyMinimize

from common_utils.minimization import run_minimization_scan, run_minimization_while
from common_utils.utils import vector2symmat
from common_utils import dneb
from common_utils.brownian import run_brownian
from jaxopt.implicit_diff import custom_root
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

f32 = jnp.float32
f64 = jnp.float64

from scipy.signal import butter, filtfilt

import os
import pickle
import sys
import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 16})
matplotlib.rcParams['text.latex.preamble']=r"\usepackage{amsmath}"
from matplotlib.legend_handler import HandlerErrorbar

# import my own modules from cluster repository
sys.path.append('/link/to/cluster/scripts/')      ### TODO ###
import mdl_model
import mdl_classification
import mdl_brownian

In [ ]:
def setup_energy_double_morse(params):
    """ Machine-readable parameters. Translation to human-readable parameters not done during optimization
         Credits: Carl Goodrich.
         Args:
             params: parameters of 1D double-well potential.
         Returns:
             energy function.
    """
    def energy_fn(r, new_params=None):
        if new_params is not None:
          epsilon0, epsilon1, sigma0, sigma1, alpha0, alpha1, A, B = new_params
        else:
          epsilon0, epsilon1, sigma0, sigma1, alpha0, alpha1, A, B = params
        r = jnp.array(r, jnp.float64)
        E0 = energy.morse(r - sigma0, sigma=0, epsilon=epsilon0, alpha=alpha0)
        E1 = energy.morse(sigma1 - r, sigma=0, epsilon=epsilon1, alpha=alpha1)
        return E0 + E1 + A + B * r
    return energy_fn

def get_energy_double_morse(params):
    params_mac = jnp.array([params['mac_epsilon0'], params['mac_epsilon1'], params['mac_sigma0'], params['mac_sigma1'],
                            params['mac_alpha0'], params['mac_alpha1'], params['mac_A'], params['mac_B']])
    params_src = jnp.array([params['src_epsilon0'], params['src_epsilon1'], params['src_sigma0'], params['src_sigma1'],
                            params['src_alpha0'], params['src_alpha1'], params['src_A'], params['src_B']])
    energy_mac_fn = setup_energy_double_morse(params_mac)
    energy_src_fn = setup_energy_double_morse(params_src)
    return energy_mac_fn, energy_src_fn

def get_energy_double_morse_with_hparams(params):
  def extract_human_params(V, r0, r1, rt):
    solver = ScipyMinimize(method='L-BFGS-B', fun=lambda r: V(r)[0],  maxiter=1000, tol=1e-12)
    #solver = LBFGS(fun=lambda r: V(r)[0],  maxiter=1000, tol=1e-12)
    res0 = solver.run(jnp.array([r0]))
    res1 = solver.run(jnp.array([r1]))
    solver = ScipyMinimize(method='L-BFGS-B', fun=lambda r: -V(r)[0],  maxiter=1000, tol=1e-12)
    #solver = LBFGS(fun=lambda r: -V(r)[0],  maxiter=1000, tol=1e-12)
    rest = solver.run(jnp.array([rt]))

    return res0[0][0], res1[0][0], rest[0][0]

  Vm, Vs = get_energy_double_morse(params)

  m0, m1, mt = extract_human_params(Vm, params['mac_sigma0'], params['mac_sigma1'], (params['mac_sigma0'] + params['mac_sigma1'])/2)
  s0, s1, st = extract_human_params(Vs, params['src_sigma0'], params['src_sigma1'], (params['src_sigma0'] + params['src_sigma1'])/2)

  return Vm, Vs, m0, m1, mt, s0, s1, st

def Vpatch(r, patch_params):
  sigma, epsilon, alpha = patch_params
  return energy.morse(r, sigma=sigma, epsilon=epsilon, alpha=alpha)

def setup_energy(params, return_human_params = True):
  if return_human_params:
    Vm, Vs, m0, m1, mt, s0, s1, st = get_energy_double_morse_with_hparams(params)
  else:
    Vm, Vs = get_energy_double_morse(params)

  default_patch_params = jnp.array([0., params['epsilon_pp'], params['alpha_pp']])

  def get_r1_r2(m,s,a):
    ms = jnp.where(m>s, m-s, s-m)

    test = -4 + jnp.exp(a * ms)
    test2 = 0.5*(jnp.exp(a * ms) - jnp.exp(a * ms / 2) * jnp.sqrt(jnp.where(test > 0, test, 1)))
    r2temp = jnp.log(jnp.where(test2>0, test2, 1)) / a
    r2 = jnp.where(test > 0, r2temp, ms / 2)
    r1 = ms - r2
    return r1, r2

  def energy_fn(R, params=None):
    m,s = jnp.ravel(R)

    if params is None:
      params_mac = None
      params_src = None
      patch_params = default_patch_params
      r1, r2 = get_r1_r2(m, s, default_patch_params[2])
    else:
      params_mac = jnp.array([params['mac_epsilon0'], params['mac_epsilon1'], params['mac_sigma0'], params['mac_sigma1'],
                              params['mac_alpha0'], params['mac_alpha1'], params['mac_A'], params['mac_B']])
      params_src = jnp.array([params['src_epsilon0'], params['src_epsilon1'], params['src_sigma0'], params['src_sigma1'],
                              params['src_alpha0'], params['src_alpha1'], params['src_A'], params['src_B']])
      patch_params = jnp.array([0., params['epsilon_pp'], params['alpha_pp']])
      r1, r2 = get_r1_r2(m, s, params['alpha_pp'])

    return Vm(m, params_mac) + Vs(s, params_src) + Vpatch(r1, patch_params) + Vpatch(r2, patch_params)

  if return_human_params:
    return energy_fn, m0, m1, mt, s0, s1, st
  else:
    return energy_fn

def update_energy_fn(energy_fn, params):
  def new_energy_fn(R):
    return energy_fn(R, params)
  return new_energy_fn

In [ ]:
def V(r, dimer_params):
    """ Model potentials Vm and Vs as double-morse potential. """
    epsilon0, epsilon1, sigma0, sigma1, alpha0, alpha1, A, B = dimer_params
    
    U0 = epsilon0 * (1 - jnp.exp(-alpha0 * ( r - sigma0)))**2 - epsilon0
    U1 = epsilon1 * (1 - jnp.exp(-alpha1 * (-r + sigma1)))**2 - epsilon1
    
    return A + B * r + U0 + U1


def get_Vm_Vs(params):
    """ Get energy potentials Vm and Vs of the Machine and Fuel Source. """
    params_mac = jnp.array([params['mac_epsilon0'], params['mac_epsilon1'],
                          params['mac_sigma0'], params['mac_sigma1'],
                          params['mac_alpha0'], params['mac_alpha1'],
                          params['mac_A'], params['mac_B']])
    params_src = jnp.array([params['src_epsilon0'], params['src_epsilon1'],
                          params['src_sigma0'], params['src_sigma1'],
                          params['src_alpha0'], params['src_alpha1'],
                          params['src_A'], params['src_B']])
    Vm = lambda r: V(r, params_mac)
    Vs = lambda s: V(s, params_src)
    return Vm, Vs

## not jit-able version
solver1 = ScipyMinimize(method='L-BFGS-B', fun=V,  maxiter=1000, tol=1e-12)
solver2 = ScipyMinimize(method='CG', fun=lambda r, dimer_params: -V(r, dimer_params),  maxiter=1000, tol=1e-12)

def get_human_params(dimer_params):
    """ Get human readable parameters for minima and saddle point from
    minimizations of the potential energy. """
    guess0 = dimer_params[2] 
    guess1 = dimer_params[3] 
    guesst = (guess0 + guess1) / 2.0
    
    res0 = solver1.run(guess0, dimer_params)
    res1 = solver1.run(guess1, dimer_params)
    rest = solver2.run(guesst, dimer_params)
    
    return res0[0], res1[0], rest[0]

def get_all_hparams(params):
    """ Get all human parameters for minima and saddle point. """
    params_mac = jnp.array([params['mac_epsilon0'], params['mac_epsilon1'],
                          params['mac_sigma0'], params['mac_sigma1'],
                            params['mac_alpha0'], params['mac_alpha1'],
                          params['mac_A'], params['mac_B']])
    params_src = jnp.array([params['src_epsilon0'], params['src_epsilon1'],
                          params['src_sigma0'], params['src_sigma1'],
                            params['src_alpha0'], params['src_alpha1'],
                          params['src_A'], params['src_B']])
    
    m0, m1, mt = get_human_params(params_mac)
    s0, s1, st = get_human_params(params_src)
    
    return m0, m1, mt, s0, s1, st

In [ ]:
color_m = '#404040'
color_s = '#4169E1'
color_pl = '#800080'
color_pr = '#dda0dd'

# State simulations Machine
Data for Machine only from ``individual_simulation_state.py`` for a given binding energy $\epsilon$.

## Analysis

In [ ]:
data_dir_mac = (r'/link/to/simulation/data/Machine/')    ### TODO ###
file = os.path.join(data_dir_mac, '../params.pickle')
with open(file, 'rb') as f:
    params = pickle.load(f)

file = os.path.join(data_dir_mac, 'global_options.pickle')
with open(file, 'rb') as f:
    global_options = pickle.load(f)

energy_fn, m0, m1, mt, s0, s1, st = setup_energy(params, True)

In [ ]:
def analyze_data(data):
    s_final = jnp.array(data['s_final'].values)
    n_trans = jnp.array(data['n_trans'].values)
    times_trans = jnp.array(data['times_trans'].values)

    if n_trans == 0:
        T = 0
        N_actions = 0
    else:
        if s_final == 0:
            times_trans = times_trans[:-1]
            N_actions = n_trans // 2 - 1

        else:
            N_actions = n_trans // 2
        T = (times_trans[-1] - times_trans[0]) * global_options['dt']
    dwell_times = jnp.ediff1d(times_trans) * global_options['dt']
    
    return N_actions, T, dwell_times

In [ ]:
job_idx = 8
data = xr.open_dataset(os.path.join(data_dir_mac, './data_{:03d}.nc'.format(job_idx)))
s_final = jnp.array(data['s_final'].values)
n_trans = jnp.array(data['n_trans'].values)
times_trans = jnp.array(data['times_trans'].values)
N_actions, T, dwell_times = analyze_data(data)
dwell_times_off = jnp.copy(dwell_times[1::2])
dwell_times_on = jnp.copy(dwell_times[::2])

print('n_trans =', n_trans)
print('Ns_actions =', jnp.mean(N_actions))
print('dwell_times = jnp.ediff1d(times_trans) =')
print(jnp.ediff1d(times_trans))

In [ ]:
unique_vals, counts = jnp.unique(jnp.ediff1d(times_trans), return_counts=True)
print(unique_vals.shape, counts)

In [ ]:
times_trans[unique_vals.shape[0]]

In [ ]:
N_rep_mac = 100
Ns_actions = jnp.zeros(N_rep_mac)
Ts = jnp.zeros(N_rep_mac)
for job_idx in range(N_rep_mac): 
    data = xr.open_dataset(os.path.join(data_dir_mac, './data_{:03d}.nc'.format(job_idx)))
    N_actions, T, dwell_times = analyze_data(data)
    Ns_actions = Ns_actions.at[job_idx].set(N_actions)
    Ts = Ts.at[job_idx].set(T)
    if job_idx == 0:
        dwell_times_off = jnp.copy(dwell_times[1::2])
        dwell_times_on = jnp.copy(dwell_times[::2])
    else:
        dwell_times_off = jnp.append(dwell_times_off, dwell_times[1::2])
        dwell_times_on = jnp.append(dwell_times_on, dwell_times[::2])

jnp.save(os.path.join(data_dir_mac, 'Ns_actions.npy'), Ns_actions)
jnp.save(os.path.join(data_dir_mac, 'Ts.npy'), Ts)
jnp.save(os.path.join(data_dir_mac, 'dwell_times_off.npy'), dwell_times_off)
jnp.save(os.path.join(data_dir_mac, 'dwell_times_on.npy'), dwell_times_on)
print('Ns_actions =', jnp.mean(Ns_actions), '+-', jnp.std(Ns_actions))
print('Ts =', jnp.mean(Ts), '+-', jnp.std(Ts))
print('dwell_times_off =', jnp.mean(dwell_times_off), '+-', jnp.std(dwell_times_off))
print('dwell_times_on =', jnp.mean(dwell_times_on), '+-', jnp.std(dwell_times_on))

In [ ]:
val_Ns_mac = jnp.nanmean(Ns_actions / Ts)
err_Ns_mac = jnp.nanstd(Ns_actions / Ts)
p_B_mac_avg = jnp.sum(dwell_times_on) / jnp.sum(Ts)
print(p_B_mac_avg)

In [ ]:
def get_prob_B_eq(params):
    kT = 1.
    Vm, Vs = get_Vm_Vs(params)
    m0, m1, mt, s0, s1, st = get_all_hparams(params)
    N_smp = 500
    m_buff = 0.5
    ms = jnp.linspace(m0-m_buff, m1+m_buff, N_smp)
    dm = ms[1]-ms[0]
    Vms = Vm(ms)
    prob_eq = jnp.exp(-Vms/kT + 2*jnp.log(ms))
    prob_eq_re = prob_eq / (jnp.sum(prob_eq) * dm)
    p_B_eq = jnp.sum(prob_eq_re[int(N_smp/2):] * dm)
    return p_B_eq
p_B_eq = get_prob_B_eq(params)
print('p_B_eq   =', p_B_eq)

p_B_data = jnp.sum(dwell_times_on) / jnp.sum(Ts)
p_B_err = jnp.sqrt((jnp.std(Ts) / jnp.mean(Ts))**2 + (jnp.std(dwell_times_on) / jnp.mean(dwell_times_on))**2) / jnp.sqrt(dwell_times_on.shape[0])
print('p_B_data =', p_B_data, '+-', p_B_err)

In [ ]:
m_buff = 0.3
ms = jnp.linspace(m0-m_buff, m1+m_buff, 1000)
dm = ms[1] - ms[0]
Vms = Vm(ms)
kT = 1.
prob_eq = jnp.exp(-Vms/kT + 2*jnp.log(ms))
prob_eq_re = prob_eq / (jnp.sum(prob_eq) * dm)
plt.plot(ms, prob_eq_re, label='eq', color='#222222', lw=2, zorder=2)
plt.xlabel(r'Machine length $m$')
plt.ylabel(r'probability $p(m)$')
plt.show()

In [ ]:
N_bins = 20
count, bin_edges = jnp.histogram(Ns_actions, bins=N_bins, density=False)
dbin = bin_edges[2] - bin_edges[1]
locs_n = bin_edges[:-1] + dbin/2
plt.bar(locs_n, count, width=2, color='#888888') 
plt.axvline(jnp.mean(Ns_actions), color='#222222', lw=3)
plt.xlabel(r'number of Machine actions', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.ylim([0,12])
# plt.savefig(os.path.join(data_dir_mac, 'N_actions.pdf'), bbox_inches='tight')
plt.show()

## Dwell times & transition rates

In [ ]:
def fit_fun(x, a, b):
    return a * jnp.exp(-b * x)

print('rate from mean    =', 1 / jnp.mean(dwell_times_off), '+-', jnp.std(dwell_times_off) / jnp.mean(dwell_times_off)**2 / jnp.sqrt(N_rep_mac))
N_bins = 100
plt.hist(dwell_times_off, bins=N_bins, color='#888888', density=False)
counts, bin_edges = jnp.histogram(dwell_times_off, bins=N_bins, density=False)
bin_center = bin_edges[:-1] + jnp.diff(bin_edges) / 2
pguess = [1, 0.001]
popt, pcov = curve_fit(fit_fun, bin_center, counts, p0=pguess)
print('rate from exp fit =', popt[-1])
plt.plot(bin_center, fit_fun(bin_center, *popt), c='#555555', lw=3)
plt.axvline(jnp.mean(dwell_times_off), color='#222222', lw=3)
plt.xlabel(r'dwell time', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.xticks((0,2500,5000,7500,10000,12500,15000), ('0','','5000','','10000','','15000'), fontsize=24)
plt.yticks(fontsize=24)
plt.title('uncharged Machine state', fontsize=24)
plt.xlim([0,16000])
# plt.savefig(os.path.join(data_dir_mac, 'dwell_times_mac_clsd.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
def fit_fun(x, a, b):
    return a * jnp.exp(-b * x)

print('rate from mean    =', 1 / jnp.mean(dwell_times_on), '+-', jnp.std(dwell_times_on) / jnp.mean(dwell_times_on)**2 / jnp.sqrt(N_rep_mac))
N_bins = 100
plt.hist(dwell_times_on, bins=N_bins, color='#888888', density=False)
counts, bin_edges = jnp.histogram(dwell_times_on, bins=N_bins, density=False)
bin_center = bin_edges[:-1] + jnp.diff(bin_edges) / 2
pguess = [1, 0.001]
popt, pcov = curve_fit(fit_fun, bin_center, counts, p0=pguess)
print('rate from exp fit =', popt[-1])
plt.plot(bin_center, fit_fun(bin_center, *popt), c='#555555', lw=3)
plt.axvline(jnp.mean(dwell_times_on), color='#222222', lw=3)
plt.xlabel(r'dwell time', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.title('charged Machine state', fontsize=24)
plt.xticks((0,100,200,300,400,500,600), ('0','','200','','400','','600'), fontsize=24)
plt.yticks(fontsize=24)
plt.xlim([0,600])
# plt.savefig(os.path.join(data_dir_mac, 'dwell_times_mac_open.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# Kramers rates
gamma = 0.1  # used in simulation
beta = 1.

Vm, Vs = get_energy_double_morse(params)
hess_mac_fn = hessian(Vm) 
m_buff = 0.3
ms = jnp.linspace(m0-m_buff, m1+m_buff, 1000)
Vms = Vm(ms)
arg_u = jnp.argmin(Vms[:300])
arg_t = int(130+jnp.argmax(Vms[130:800]))
arg_c = int(500+jnp.argmin(Vms[500:]))
dE_uc = Vms[arg_t] - Vms[arg_u]
dE_cu = Vms[arg_t] - Vms[arg_c]
# print(dE_uc, dE_cu)

# factor of 2 in the (positive) eigenvalues of the Hessian V''(x2-x1) = V(x)*[[1,-1],[-1,1]]; V'(x2-x1) = V'(x)*[-1,1] due to coordinate transformation x := x2-x1?
k_uc = jnp.sqrt(2*hess_mac_fn(ms[arg_u])) * jnp.sqrt(-2*hess_mac_fn(ms[arg_t])) / (2*jnp.pi*gamma) * jnp.exp(-beta * (dE_uc - 2/beta*jnp.log(ms[arg_t] / ms[arg_u])))
k_cu = jnp.sqrt(2*hess_mac_fn(ms[arg_c])) * jnp.sqrt(-2*hess_mac_fn(ms[arg_t])) / (2*jnp.pi*gamma) * jnp.exp(-beta * (dE_cu - 2/beta*jnp.log(ms[arg_t] / ms[arg_c])))
print(k_uc, k_cu)

k_uc_md = 0.001136
k_cu_md = 0.0450
print(k_uc_md / k_cu_md)

In [ ]:
p_u = jnp.exp(Vms[arg_u]) / (jnp.exp(Vms[arg_u]) + jnp.exp(Vms[arg_c]))
p_c = jnp.exp(Vms[arg_c]) / (jnp.exp(Vms[arg_u]) + jnp.exp(Vms[arg_c]))
print(p_u,  p_c)
print(p_u / p_c)

# State simulations Machine & Source
Data for coupled system from ``coupled_simulation_state.py`` for a given binding energy $\epsilon$.

In [ ]:
def get_Ecapm(params):
    Vm, Vs = get_energy_double_morse(params)
    energy_fn, m0, m1, mt, s0, s1, st = setup_energy(params, True)
    Ecap_m = Vm(m1)-Vm(m0)
    return Ecap_m

def get_Ecaps(params):
    Vm, Vs = get_energy_double_morse(params)
    energy_fn, m0, m1, mt, s0, s1, st = setup_energy(params, True)
    Ecap_s = Vs(s0)-Vs(s1)
    return Ecap_s

delta_Em = get_Ecapm(params)
delta_Es = get_Ecaps(params)

def get_prob_B_eq(params):
    kT = 1.
    m_buff = 0.3
    Vm, Vs = get_Vm_Vs(params)
    m0, m1, mt, s0, s1, st = get_all_hparams(params)
    ms = jnp.linspace(m0-m_buff, m1+m_buff, 500)
    dm = ms[1]-ms[0]
    Vms = Vm(ms)
    prob_eq = jnp.exp(-Vms/kT + 2*jnp.log(ms))
    prob_eq_re = prob_eq / (jnp.sum(prob_eq)  * dm)
    p_B_eq = jnp.sum(prob_eq_re[250:] * dm)
    return p_B_eq
p_B_eq = get_prob_B_eq(params)

## Analysis

In [ ]:
def analyze_data(data):
    s_final = jnp.array(data['s_final'].values)
    n_trans = jnp.array(data['n_trans'].values)
    times_trans = jnp.array(data['times_trans'].values)
    N_re = jnp.array(data['N_re'].values)
    t_use0 = jnp.array(data['t_use0'].values)
    t_use1 = jnp.array(data['t_use1'].values)

    if n_trans == 0:
        T = 0
        N_actions = 0
        N_re_data = 0
        t_use0_data = 0
        t_use1_data = 0
    else:
        if s_final == 0:
            times_trans = times_trans[:-1]
            N_re_data = jnp.sum(N_re[1:-2])
            t_use0_data = jnp.sum(t_use0[1:-2]) * global_options['dt']
            t_use1_data = jnp.sum(t_use1[1:-2]) * global_options['dt']
            N_actions = n_trans // 2 - 1
            
        else:
            N_re_data = jnp.sum(N_re[1:-1])
            t_use0_data = jnp.sum(t_use0[1:-1]) * global_options['dt']
            t_use1_data = jnp.sum(t_use1[1:-1]) * global_options['dt']
            N_actions = n_trans // 2
    
        T = (times_trans[-1] - times_trans[0]) * global_options['dt']
    dwell_times = jnp.ediff1d(times_trans) * global_options['dt']
    return N_actions, dwell_times, T, N_re_data, t_use0_data, t_use1_data

In [ ]:
data_dir = (r'/link/to/data/coupled/eps15/')      ### TODO ###
file = os.path.join(data_dir, 'params.pickle')
with open(file, 'rb') as f:
    params = pickle.load(f)
file = os.path.join(data_dir, 'global_options.pickle')
with open(file, 'rb') as f:
    global_options = pickle.load(f)

In [ ]:
N_rep = 200
for job_idx in range(N_rep):
    data = xr.open_dataset(os.path.join(data_dir, './data_{:03d}.nc'.format(job_idx)))
    N_actions, dwell_times, T, N_re, t_use0, t_use1 = analyze_data(data)
    if job_idx == 0:
        Ns_actions = jnp.copy(N_actions)
        Ns_re = jnp.copy(N_re)
        dwell_times_off = jnp.copy(dwell_times[1::2])
        dwell_times_on = jnp.copy(dwell_times[::2])
        Ts = jnp.copy(T)
        ts_use0 = jnp.copy(t_use0)
        ts_use1 = jnp.copy(t_use1)
    else:
        Ns_actions = jnp.append(Ns_actions, N_actions)
        Ns_re = jnp.append(Ns_re, N_re)
        dwell_times_off = jnp.append(dwell_times_off, dwell_times[1::2])
        dwell_times_on = jnp.append(dwell_times_on, dwell_times[::2])
        Ts = jnp.append(Ts, T)
        ts_use0 = jnp.append(ts_use0, t_use0)
        ts_use1 = jnp.append(ts_use1, t_use1)
jnp.save(os.path.join(data_dir, 'Ns_actions.npy'), Ns_actions)
jnp.save(os.path.join(data_dir, 'Ns_re.npy'), Ns_re)
jnp.save(os.path.join(data_dir, 'Ts.npy'), Ts)
jnp.save(os.path.join(data_dir, 'ts_use0.npy'), ts_use0)
jnp.save(os.path.join(data_dir, 'ts_use1.npy'), ts_use1)
print('Ns_actions =', jnp.mean(Ns_actions), '+-', jnp.std(Ns_actions))
print('Ts =', jnp.mean(Ts), '+-', jnp.std(Ts))
print('dwell_times_off =', jnp.mean(dwell_times_off), '+-', jnp.std(dwell_times_off))
print('dwell_times_on =', jnp.mean(dwell_times_on), '+-', jnp.std(dwell_times_on))

In [ ]:
p_B_mac_avg = jnp.sum(dwell_times_on) / jnp.sum(Ts)
print(p_B_mac_avg)

In [ ]:
N_eff = jnp.count_nonzero(Ts)    
val_Ns = jnp.nanmean(Ns_actions / Ts)
err_Ns = jnp.nanstd(Ns_actions / Ts) / jnp.sqrt(N_eff)
val_Ns_re = jnp.nanmean(Ns_re / Ts)
err_Ns_re = jnp.nanstd(Ns_re / Ts) / jnp.sqrt(N_eff)
val_ts0 = jnp.nanmean(ts_use0 / Ts)
err_ts0 = jnp.nanstd(ts_use0 / Ts) / jnp.sqrt(N_eff)
val_ts1 = jnp.nanmean(ts_use1 / Ts)
err_ts1 = jnp.nanstd(ts_use1 / Ts) / jnp.sqrt(N_eff)

In [ ]:
performance0_avg = val_ts0 / p_B_eq
performance0_err = err_ts0 / p_B_eq
performance1_avg = val_ts1 / p_B_eq
performance1_err = err_ts1 / p_B_eq
power_avg = (val_Ns - val_Ns_mac) * delta_Em
power_err = jnp.sqrt(err_Ns**2 + err_Ns_mac**2) * delta_Em
efficiency_avg = (val_Ns - val_Ns_mac) / val_Ns_re * delta_Em / delta_Es
efficiency_err = jnp.sqrt((jnp.sqrt(err_Ns**2 + err_Ns_mac**2) / (val_Ns - val_Ns_mac))**2 + (err_Ns_re / val_Ns_re)**2) * jnp.abs(efficiency_avg)

In [ ]:
print('N_actions =', jnp.mean(Ns_actions), '+-', jnp.std(Ns_actions) / jnp.sqrt(N_rep))
print('N_re =', jnp.mean(Ns_re), '+-', jnp.std(Ns_re) / jnp.sqrt(N_rep))
print('T =', jnp.mean(Ts), '+-', jnp.std(Ts) / jnp.sqrt(N_rep))
print('performance0 =', performance0_avg, '+-', performance0_err)
print('performance1 =', performance1_avg, '+-', performance1_err)
print('power =', power_avg, '+-', power_err)
print('efficiency =', efficiency_avg, '+-', efficiency_err)

In [ ]:
N_bins = 16
plt.hist(Ns_actions, bins=N_bins, density=False)
plt.axvline(jnp.mean(Ns_actions), color='#222222', lw=3)
plt.xlabel(r'number of Machine actions', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
# plt.savefig(os.path.join(data_dir, 'N_actions.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
def fit_fun(x, a, b):
    return a * jnp.exp(-b * x)

print('rate from mean    =', 1 / jnp.mean(dwell_times_off), '+-', jnp.std(dwell_times_off) / jnp.mean(dwell_times_off)**2 / jnp.sqrt(dwell_times_off.shape[0]))
N_bins = 30
plt.hist(dwell_times_off, bins=N_bins, density=False)
counts, bin_edges = jnp.histogram(dwell_times_off, bins=N_bins, density=False)
bin_center = bin_edges[:-1] + jnp.diff(bin_edges) / 2
pguess = [1, 0.001]
popt, pcov = curve_fit(fit_fun, bin_center, counts, p0=pguess)
print('rate from exp fit =', popt[-1])
plt.plot(bin_center, fit_fun(bin_center, *popt), c='#999999', lw=3)

plt.vlines(jnp.mean(dwell_times_off), 0, jnp.max(counts), color='#222222', lw=3)
plt.xlabel(r'dwell time', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.title('closed Machine state', fontsize=24)
plt.xlim([0,600])
# plt.savefig(os.path.join(data_dir, 'dwell_times_mac_clsd.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
def fit_fun(x, a, b):
    return a * jnp.exp(-b * x)

print('rate from mean    =', 1 / jnp.mean(dwell_times_on), '+-', jnp.std(dwell_times_on) / jnp.mean(dwell_times_on)**2 / jnp.sqrt(dwell_times_on.shape[0]))
N_bins = 30
plt.hist(dwell_times_on, bins=N_bins, density=False)
counts, bin_edges = jnp.histogram(dwell_times_on, bins=N_bins, density=False)
bin_center = bin_edges[:-1] + jnp.diff(bin_edges) / 2
pguess = [1, 0.001]
popt, pcov = curve_fit(fit_fun, bin_center, counts, p0=pguess)
print('rate from exp fit =', popt[-1])
plt.plot(bin_center, fit_fun(bin_center, *popt), c='#999999', lw=3)

plt.vlines(jnp.mean(dwell_times_on), 0, jnp.max(counts), color='#222222', lw=3)
plt.xlabel(r'dwell time', fontsize=24)
plt.ylabel('count', fontsize=24)
plt.title('open Machine state', fontsize=24)
plt.xticks((0,100,200,300,400,500,600), ('0','','200','','400','','600'), fontsize=24)
plt.yticks(fontsize=24)
plt.xlim([0,600])
# plt.savefig(os.path.join(data_dir, 'dwell_times_mac_open.pdf'), bbox_inches='tight')
plt.show()

# Loop over data (epsilon)
Data for coupled system Machine & Source from ``coupled_simulation_state.py`` for a range of binding energies $\epsilon=[0,1,\dots,25]$ that are saved in subfolders.

In [ ]:
colors = ['#006600', '#feb24c', '#dd1c77']

data_dir_mac = '/link/to/data/Machine/'    ### TODO ###
data_dir = '/link/to/data/coupled/'        ### TODO ###
data_dirs = []
for eps in range(26):
    data_dirs.append(os.path.join(data_dir, 'eps{:02d}'.format(eps)))

file = os.path.join(data_dirs[0], 'global_options.pickle')
with open(file, 'rb') as f:
    global_options = pickle.load(f)
file = os.path.join(data_dirs[0], 'params.pickle')
with open(file, 'rb') as f:
    params = pickle.load(f)

def get_delta_Es(params):
    Vm, Vs = get_Vm_Vs(params)
    delta_Em = Vm(params['mac_sigma1']) - Vm(params['mac_sigma0'])
    delta_Es = Vs(params['src_sigma0']) - Vs(params['src_sigma1'])
    return delta_Em, delta_Es
    
delta_Em, delta_Es = get_delta_Es(params)

def get_prob_B_eq(params):
    kT = 1.
    m_buff = 0.3
    Vm, Vs = get_Vm_Vs(params)
    m0, m1, mt, s0, s1, st = get_all_hparams(params)
    ms = jnp.linspace(m0-m_buff, m1+m_buff, 500)
    dm = ms[1]-ms[0]
    Vms = Vm(ms)
    prob_eq = jnp.exp(-Vms/kT + 2*jnp.log(ms))
    prob_eq_re = prob_eq / (jnp.sum(prob_eq)  * dm)
    p_B_eq = jnp.sum(prob_eq_re[250:] * dm)
    return p_B_eq
    
p_B_eq = get_prob_B_eq(params)

In [ ]:
print(delta_Em / delta_Es)

In [ ]:
N_data = len(data_dirs)
performances0_avg = jnp.zeros(N_data)
performances0_err = jnp.zeros(N_data)
performances1_avg = jnp.zeros(N_data)
performances1_err = jnp.zeros(N_data)
powers_avg = jnp.zeros(N_data)
powers_err = jnp.zeros(N_data)
efficiencies_avg = jnp.zeros(N_data)
efficiencies_err = jnp.zeros(N_data)
Ns_actions_avg = jnp.zeros(N_data)
Ns_actions_err = jnp.zeros(N_data)
Ns_re_avg = jnp.zeros(N_data)
Ns_re_err = jnp.zeros(N_data)
N_rep = 200

# Machine only
Ns_actions_mac = jnp.load(os.path.join(data_dir_mac, 'Ns_actions.npy'))
Ts_mac = jnp.load(os.path.join(data_dir_mac, 'Ts.npy'))
dwell_times_on_mac = jnp.load(os.path.join(data_dir_mac, 'dwell_times_on.npy'))
dwell_times_off_mac = jnp.load(os.path.join(data_dir_mac, 'dwell_times_off.npy'))
val_Ns_mac = jnp.nanmean(Ns_actions_mac / Ts_mac)
err_Ns_mac = jnp.nanstd(Ns_actions_mac / Ts_mac)
p_B_mac_avg = jnp.sum(dwell_times_on_mac) / jnp.sum(Ts_mac)

# Machine & Source
for n, data_dir in enumerate(data_dirs):
    Ns_actions = jnp.load(os.path.join(data_dir, 'Ns_actions.npy'))
    Ns_re = jnp.load(os.path.join(data_dir, 'Ns_re.npy'))
    Ts = jnp.load(os.path.join(data_dir, 'Ts.npy'))
    ts_use0 = jnp.load(os.path.join(data_dir, 'ts_use0.npy'))
    ts_use1 = jnp.load(os.path.join(data_dir, 'ts_use1.npy'))
    
    N_eff = jnp.count_nonzero(Ts)    
    val_Ns = jnp.nanmean(Ns_actions / Ts)
    err_Ns = jnp.nanstd(Ns_actions / Ts)
    val_Ns_re = jnp.nanmean(Ns_re / Ts)
    err_Ns_re = jnp.nanstd(Ns_re / Ts)
    val_ts0 = jnp.nanmean(ts_use0 / Ts)
    err_ts0 = jnp.nanstd(ts_use0 / Ts)
    val_ts1 = jnp.nanmean(ts_use1 / Ts)
    err_ts1 = jnp.nanstd(ts_use1 / Ts)
    
    performance0_avg = val_ts0 / p_B_mac_avg
    performance0_err = err_ts0 / p_B_mac_avg
    performance1_avg = val_ts1 / p_B_mac_avg
    performance1_err = err_ts1 / p_B_mac_avg
    power_avg = (val_Ns - val_Ns_mac) * delta_Em
    power_err = jnp.sqrt(err_Ns**2 + err_Ns_mac**2) * delta_Em
    efficiency_avg = (val_Ns - val_Ns_mac) / val_Ns_re * delta_Em / delta_Es
    efficiency_err = jnp.sqrt((jnp.sqrt(err_Ns**2 + err_Ns_mac**2) / (val_Ns - val_Ns_mac))**2 + (err_Ns_re / val_Ns_re)**2) * jnp.abs(efficiency_avg)

    performances0_avg = performances0_avg.at[n].set(performance0_avg)
    performances0_err = performances0_err.at[n].set(performance0_err)
    performances1_avg = performances1_avg.at[n].set(performance1_avg)
    performances1_err = performances1_err.at[n].set(performance1_err)
    powers_avg = powers_avg.at[n].set(power_avg)
    powers_err = powers_err.at[n].set(power_err)
    efficiencies_avg = efficiencies_avg.at[n].set(efficiency_avg)
    efficiencies_err = efficiencies_err.at[n].set(efficiency_err)
    Ns_actions_avg = Ns_actions_avg.at[n].set(jnp.nanmean(Ns_actions))
    Ns_actions_err = Ns_actions_err.at[n].set(jnp.nanstd(Ns_actions))
    Ns_re_avg = Ns_re_avg.at[n].set(jnp.nanmean(Ns_re))
    Ns_re_err = Ns_re_err.at[n].set(jnp.nanstd(Ns_re))

jnp.save(os.path.join(data_dir, 'Ns_actions_avg.npy'), Ns_actions_avg)
jnp.save(os.path.join(data_dir, 'Ns_actions_err.npy'), Ns_actions_err)
jnp.save(os.path.join(data_dir, 'Ns_re_avg.npy'), Ns_re_avg)
jnp.save(os.path.join(data_dir, 'Ns_re_err.npy'), Ns_re_err)
jnp.save(os.path.join(data_dir, 'performances0_avg.npy'), performances0_avg)
jnp.save(os.path.join(data_dir, 'performances0_err.npy'), performances0_err)
jnp.save(os.path.join(data_dir, 'performances1_avg.npy'), performances1_avg)
jnp.save(os.path.join(data_dir, 'performances1_err.npy'), performances1_err)
jnp.save(os.path.join(data_dir, 'powers_avg.npy'), powers_avg)
jnp.save(os.path.join(data_dir, 'powers_err.npy'), powers_err)
jnp.save(os.path.join(data_dir, 'efficiencies_avg.npy'), efficiencies_avg)
jnp.save(os.path.join(data_dir, 'efficiencies_err.npy'), efficiencies_err)

In [ ]:
epsilon = jnp.arange(0,26,1)

fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
plt.errorbar(epsilon, performances0_avg, yerr=performances0_err, fmt='s-', color=colors[0], lw=3, capsize=6)
plt.axhline(1, color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'performance $p^\prime$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(jnp.arange(0,9,1), ('0','','2','','4','','6','','8'), fontsize=24)
plt.xlim([-0.5,25.5])
plt.ylim([0,8])
# plt.savefig(os.path.join(data_dir, 'performance0.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
performances0_err[jnp.argmax(performances0_avg)]

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
plt.errorbar(epsilon, performances1_avg, yerr=performances1_err, fmt='s-', color=colors[0], lw=3, capsize=6)
plt.axhline(1, color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'performance $p$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(jnp.arange(0,21,2.5), ('0','','5','','10','','15','','20'), fontsize=24)
plt.xlim([-0.5,25.5])
plt.ylim([0,20])
# plt.savefig(os.path.join(data_dir, 'performance1.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
performances1_err[jnp.argmax(performances1_avg)]

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
ax.tick_params(which='minor', right=True)
plt.errorbar(epsilon[9:22], powers_avg[9:22], yerr=powers_err[9:22], fmt='s-', color=colors[0], lw=3, capsize=6)
plt.hlines(0, epsilon[0], epsilon[-1], color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'power $\mathcal{P}$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.yscale('log')
plt.xlim([-0.5,25.5])
plt.ylim([5e-5,1e-1])
# plt.savefig(os.path.join(data_dir, 'power.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
powers_err[jnp.argmax(powers_avg)]

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
ax.tick_params(which='minor', right=True)
plt.errorbar(epsilon[9:22], efficiencies_avg[9:22], yerr=efficiencies_err[9:22], fmt='s-', color=colors[0], lw=3, capsize=6)
plt.hlines(0, epsilon[0], epsilon[-1], color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'efficiency $\eta$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.yscale('log')
plt.xlim([-0.5,25.5])
plt.ylim([1e-5, 1e-1])
# plt.savefig(os.path.join(data_dir, 'efficiency.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
efficiencies_err[jnp.argmax(efficiencies_avg)]

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
plt.errorbar(epsilon, Ns_actions_avg, yerr=Ns_actions_err, fmt='s-', color=colors[0], lw=3, capsize=6)
plt.plot(epsilon, jnp.repeat(jnp.mean(Ns_actions_mac), N_data), color=colors[0], lw=2)
plt.fill_between(epsilon, jnp.mean(Ns_actions_mac) + jnp.std(Ns_actions_mac), jnp.mean(Ns_actions_mac) - jnp.std(Ns_actions_mac), alpha=0.2, color=colors[0]) 
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'$N_\text{actions}$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(jnp.arange(0,3600,500), ('0','','1000','','2000','','3000',''), fontsize=24)
plt.xlim([-0.5,25.5])
# plt.savefig(os.path.join(data_dir, 'N_actions.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
ax.tick_params(which='minor', right=True)
plt.errorbar(epsilon, Ns_re_avg, yerr=Ns_re_err, fmt='s-', color=colors[0], lw=3, capsize=6)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'$N_\text{re}$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.yscale('log')
plt.xlim([-0.5,25.5])
plt.ylim([1,7e4])
# plt.savefig(os.path.join(data_dir, 'N_re.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
s_list = jnp.linspace(0, 4.5, 500)
Vm, Vs = get_energy_double_morse(params)
Vm_list = Vm(s_list)
Vs_list = Vs(s_list)

plt.plot(s_list, Vm_list - Vm_list[jnp.argmin(Vm_list[:200])], lw=3, color=color_m, zorder=2)
plt.plot(s_list, Vs_list - Vs_list[jnp.argmin(Vs_list[:200])], lw=3, color=color_s, zorder=2)
plt.xlabel(r'dimer length $m$', fontsize=26)
plt.ylabel(r'energy', fontsize=26)
plt.xticks(fontsize=26)
plt.yticks((-40,-30,-20,-10,0,10,20), (r'$-40$','','$-20$','','$0$','','$20$'), fontsize=24)
plt.xlim([0,4.5])
plt.ylim([-45,20])
# plt.savefig(os.path.join(data_dir, 'energy_profiles.pdf'), bbox_inches='tight')
plt.show()

# Comparison figure

In [ ]:
## load second dataset
data_dir2 = '/link/to/data/coupled_comp/'

Ns_cycles_avg_d2 = jnp.load(os.path.join(data_dir2, 'Ns_cycles_avg.npy'))
Ns_cycles_err_d2 = jnp.load(os.path.join(data_dir2, 'Ns_cycles_err.npy'))
Ns_re_avg_d2 = jnp.load(os.path.join(data_dir2, 'Ns_re_avg.npy'))
Ns_re_err_d2 = jnp.load(os.path.join(data_dir2, 'Ns_re_err.npy'))
performances0_avg_d2 = jnp.load(os.path.join(data_dir2, 'performances0_avg.npy'))
performances0_err_d2 = jnp.load(os.path.join(data_dir2, 'performances0_err.npy'))
performances1_avg_d2 = jnp.load(os.path.join(data_dir2, 'performances1_avg.npy'))
performances1_err_d2 = jnp.load(os.path.join(data_dir2, 'performances1_err.npy'))
powers_avg_d2 = jnp.load(os.path.join(data_dir2, 'powers_avg.npy'))
powers_err_d2 = jnp.load(os.path.join(data_dir2, 'powers_err.npy'))
efficiencies_avg_d2 = jnp.load(os.path.join(data_dir2, 'efficiencies_avg.npy'))
efficiencies_err_d2 = jnp.load(os.path.join(data_dir2, 'efficiencies_err.npy'))

colors = ['#006600', '#feb24c', '#dd1c77']

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
plt.errorbar(epsilon, performances1_avg_d2[:-5], yerr=performances1_err_d2[:-5], fmt='D-', color=colors[2], lw=3, capsize=6, label='AD')
plt.errorbar(epsilon, performances1_avg, yerr=performances1_err, fmt='s-', color=colors[1], lw=3, capsize=6, label='AD+PR')
plt.axhline(1, color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'performance $p$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(jnp.arange(0,21,2.5), ('0','','5','','10','','15','','20'), fontsize=24)
plt.xlim([-0.5,25.5])
plt.ylim([-0.3,20])
plt.legend(loc='upper left', borderpad=0.3, labelspacing=0.3, handlelength=1.3, handletextpad=0.4, fontsize=24)
# plt.savefig(os.path.join(data_dir, 'comp_performance.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
ax.tick_params(which='minor', right=True)
plt.errorbar(epsilon[10:22], powers_avg_d2[10:22], yerr=powers_err_d2[10:22], fmt='D-', color=colors[2], lw=3, capsize=6)
plt.errorbar(epsilon[9:22], powers_avg[9:22], yerr=powers_err[9:22], fmt='s-', color=colors[1], lw=3, capsize=6)
plt.hlines(0, epsilon[0], epsilon[-1], color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'power $\mathcal{P}$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.yscale('log')
plt.xlim([-0.5,25.5])
plt.ylim([5e-5,1e-1])
# plt.savefig(os.path.join(data_dir, 'comp_power.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
ax.tick_params(which='minor', right=True)
plt.errorbar(epsilon[10:22], efficiencies_avg_d2[10:22], yerr=efficiencies_err_d2[10:22], fmt='D-', color=colors[2], lw=3, capsize=6)
plt.errorbar(epsilon[9:22], efficiencies_avg[9:22], yerr=efficiencies_err[9:22], fmt='s-', color=colors[1], lw=3, capsize=6)
plt.hlines(0, epsilon[0], epsilon[-1], color='#222222', lw=2, zorder=-1)
plt.xlabel(r'binding energy $\epsilon$', fontsize=24)
plt.ylabel(r'efficiency $\eta$', fontsize=24)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.yscale('log')
plt.xlim([-0.5,25.5])
plt.ylim([1e-5, 1e-1])
# plt.savefig(os.path.join(data_dir, 'comp_efficiency.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
s_list = jnp.linspace(0, 4.5, 500)
Vm, Vs = get_energy_double_morse(params)
Vm_list = Vm(s_list)
Vs_list = Vs(s_list)

fig, ax = plt.subplots()
ax.tick_params(top=True, right=True)
plt.plot(s_list, Vm_list_prev - Vm_list_prev[jnp.argmin(Vm_list_prev[:200])], lw=3, ls='--', color=color_m, zorder=2, label='AD')
plt.plot(s_list, Vs_list_prev - Vs_list_prev[jnp.argmin(Vs_list_prev[:200])], lw=3, ls='--', color=color_s, zorder=2)
plt.plot(s_list, Vm_list - Vm_list[jnp.argmin(Vm_list[:200])], lw=3, color=color_m, zorder=2, label='AD+PR')
plt.plot(s_list, Vs_list - Vs_list[jnp.argmin(Vs_list[:200])], lw=3, color=color_s, zorder=2)
plt.xlabel(r'dimer length', fontsize=26)
plt.ylabel(r'energy profile', fontsize=26)
plt.xticks(fontsize=26)
plt.yticks((-50,-40,-30,-20,-10,0,10,20), ('',r'$-40$','','$-20$','','$0$','','$20$'), fontsize=24)
plt.xlim([0,4.5])
plt.ylim([-50,20])
plt.legend(loc='lower left', borderpad=0.3, labelspacing=0.3, handlelength=1.3, handletextpad=0.4, fontsize=24)
# plt.savefig(os.path.join(data_dir, 'comp_energy_profiles.pdf'), bbox_inches='tight')
plt.show()

# End